In [ ]:
import json        # to read and write .ipynb files (they are JSON format)
import base64      # to convert images to text so they can be stored inside the notebook
import os          # to navigate folders and check if files exist
import glob        # to find all files matching a pattern like *.ipynb
import subprocess  # to run terminal commands like git from inside Python
import re          # to search for image patterns inside markdown text

# ─── STEP 1: SET YOUR REPO ROOT ───────────────────────────────────────────────
# This tells Python where your project lives on your computer
# All file paths below will be relative to this folder
os.chdir("D:\\Project_2026\\data-science-learning-journey")

# ─── STEP 2: DEFINE YOUR TOPICS ───────────────────────────────────────────────
# Each entry is: (topic_id, display_title, folder_path, images_folder, icon_label, icon_color)
# Add a new entry here whenever you start a new topic like EDA
TOPICS = [
    {
        "id": "python",
        "title": "1.0 Python",
        "subtitle": "Syntax, OOP, Modules and more",
        "icon": "Py",
        "icon_bg": "#dbeafe",       # light blue background for icon
        "icon_color": "#1d4ed8",    # dark blue text for icon
        "folder": "Topics/1.0-Python",
        "images_folder": None,      # Python has no images folder
        "notebooks": [
            ("1.01", "Syntax, Semantics and Operators", "1.01-Syntax-Semantics-Operators"),
            ("1.02", "Control Flow, Data Structures and Functions", "1.02-Control-Flow-Data-Structures-Functions"),
            ("1.03", "Object Oriented Programming", "1.03-Object-Oriented-Programming-Design"),
            ("1.04", "Advanced Modules and Libraries", "1.04-Advanced-Modules-Libraries"),
            ("1.05", "Exercises", "1.05-Excercises"),
        ]
    },
    {
        "id": "dsa",
        "title": "2.0 Data Structures and Algorithms",
        "subtitle": "Arrays, Trees, Graphs and more",
        "icon": "DS",
        "icon_bg": "#dcfce7",       # light green background for icon
        "icon_color": "#15803d",    # dark green text for icon
        "folder": "Topics/2.0-Data_Structures",
        "images_folder": "Topics/2.0-Data_Structures/DS_Images",  # where images are stored
        "notebooks": [
            ("2.01", "Data Structure Overview", "2.01-Data-Structure-Overview"),
            ("2.02", "Big O Notation and Recursion", "2.02-The-Big-O-Notion-Recursion"),
            ("2.03", "Arrays and Hash Tables", "2.03-Arrays-HashTables"),
            ("2.04", "Singly Linked Lists", "2.04-Singly-Linked-Lists"),
            ("2.05", "Doubly Linked Lists", "2.05-Doubly-Linked-Lists"),
            ("2.06", "Stacks and Queues", "2.06-Stacks-Queues"),
            ("2.07", "Binary Search Trees", "2.07-Binary-Search-Trees"),
            ("2.08", "Tree Traversal", "2.08-Tree-Traversal"),
            ("2.09", "Heaps", "2.09-Heaps"),
            ("2.10", "Graphs", "2.10-Graphs"),
            ("2.11", "Sorting", "2.11-Sorting"),
        ]
    },
    # ── TO ADD EDA LATER, UNCOMMENT AND FILL THIS BLOCK ──
    # {
    #     "id": "eda",
    #     "title": "3.0 Exploratory Data Analysis",
    #     "subtitle": "Pandas, Matplotlib and more",
    #     "icon": "EDA",
    #     "icon_bg": "#fef9c3",
    #     "icon_color": "#854d0e",
    #     "folder": "Topics/3.0-EDA",
    #     "images_folder": "Topics/3.0-EDA/EDA_Images",
    #     "notebooks": [
    #         ("3.01", "Introduction to EDA", "3.01-Intro-EDA"),
    #     ]
    # },
]

# ─── STEP 3: EMBED IMAGES INTO NOTEBOOKS ──────────────────────────────────────
# GitHub cannot load images from file paths, so we embed them directly
# into the notebook as base64 text using the "attachment" system

def embed_images(topic):
    # Skip topics that have no images folder defined
    if not topic["images_folder"]:
        return

    images_folder = topic["images_folder"]
    folder = topic["folder"]

    # Loop through every .ipynb file in this topic's folder
    for nb_file in glob.glob(f"{folder}/*.ipynb"):

        # Read the notebook file as JSON
        with open(nb_file, "r", encoding="utf-8") as f:
            nb = json.load(f)

        changed = False  # track if we made any changes to this notebook

        # Loop through every cell in the notebook
        for cell in nb["cells"]:
            # Only process markdown cells (code cells don't have images)
            if cell["cell_type"] != "markdown":
                continue

            # Join all lines of the cell into one string to search it
            src = "".join(cell["source"])

            # Find all image filenames referenced in this cell
            # This regex looks for patterns like ![](DS_Images/filename.png)
            images_found = re.findall(r'!\[.*?\]\((?:.*?[/\\])(.+?)\)', src)

            if not images_found:
                continue  # no images in this cell, skip it

            attachments = {}  # will store the base64 image data

            for img_name in images_found:
                img_path = os.path.join(images_folder, img_name)

                # Check the image file actually exists before trying to open it
                if not os.path.exists(img_path):
                    print(f"  Warning - image not found: {img_path}")
                    continue

                # Read the image file as binary and convert to base64 text
                # base64 turns binary image data into plain text that can be stored in JSON
                with open(img_path, "rb") as f:
                    img_base64 = base64.b64encode(f.read()).decode()

                # Store the base64 image data under the image filename
                attachments[img_name] = {"image/png": img_base64}

                # Update the markdown reference from a file path to an attachment reference
                # Before: ![](DS_Images/chart.png)
                # After:  ![](attachment:chart.png)
                src = re.sub(
                    r'!\[(.*?)\]\((?:.*?[/\\])' + re.escape(img_name) + r'\)',
                    f'![\\1](attachment:{img_name})',
                    src
                )

            # Save the attachments and updated source back into the cell
            cell["attachments"] = attachments
            cell["source"] = [src]
            changed = True

        # Only write the file if we actually changed something
        if changed:
            with open(nb_file, "w", encoding="utf-8") as f:
                json.dump(nb, f, indent=1)
            print(f"  Patched: {os.path.basename(nb_file)}")

# ─── STEP 4: BUILD THE HOMEPAGE ───────────────────────────────────────────────
# Creates index.html — the main landing page with topic cards

def build_homepage():
    # Build the topic cards section dynamically from TOPICS list
    topic_cards = ""
    for topic in TOPICS:
        topic_cards += f"""
        <a href="{topic['id']}.html" class="topic-card">
            <div class="topic-icon" style="background:{topic['icon_bg']}; color:{topic['icon_color']}">
                {topic['icon']}
            </div>
            <div>
                <p class="topic-title">{topic['title']}</p>
                <p class="topic-sub">{topic['subtitle']}</p>
            </div>
            <span class="arrow">›</span>
        </a>"""

    # The full HTML for the homepage
    html = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Praveen's Data Science Journey</title>
    <style>
        body {{ font-family: Arial, sans-serif; max-width: 700px; margin: 0 auto; padding: 40px 20px; background: #f9fafb; }}
        h1 {{ font-size: 24px; font-weight: 600; color: #111; margin: 0 0 4px; }}
        .subtitle {{ color: #666; margin: 0 0 32px; font-size: 15px; }}
        .topic-card {{ display: flex; align-items: center; gap: 16px; padding: 16px 20px; border: 1px solid #e5e7eb; border-radius: 12px; background: white; margin-bottom: 12px; text-decoration: none; color: inherit; }}
        .topic-card:hover {{ border-color: #9ca3af; background: #f9fafb; }}
        .topic-icon {{ width: 44px; height: 44px; border-radius: 10px; display: flex; align-items: center; justify-content: center; font-weight: 600; font-size: 13px; flex-shrink: 0; }}
        .topic-title {{ font-weight: 600; font-size: 15px; color: #111; margin: 0 0 3px; }}
        .topic-sub {{ font-size: 13px; color: #6b7280; margin: 0; }}
        .arrow {{ margin-left: auto; color: #9ca3af; font-size: 20px; }}
    </style>
</head>
<body>
    <h1>Praveen's Data Science Journey</h1>
    <p class="subtitle">Click a topic to explore notebooks</p>
    {topic_cards}
</body>
</html>"""

    # Write the homepage to the docs folder
    with open("docs/index.html", "w", encoding="utf-8") as f:
        f.write(html)
    print("Built: index.html")

# ─── STEP 5: BUILD TOPIC LANDING PAGES ────────────────────────────────────────
# Creates one landing page per topic e.g. python.html, dsa.html

def build_topic_pages():
    for topic in TOPICS:
        # Build the list of notebook links for this topic
        notebook_links = ""
        for num, title, filename in topic["notebooks"]:
            notebook_links += f"""
            <a href="{filename}.html" class="nb-card">
                <span class="nb-num">{num}</span>
                <span class="nb-title">{title}</span>
                <span class="arrow">›</span>
            </a>"""

        html = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{topic['title']} - Praveen's DS Journey</title>
    <style>
        body {{ font-family: Arial, sans-serif; max-width: 700px; margin: 0 auto; padding: 40px 20px; background: #f9fafb; }}
        .back {{ color: #6b7280; text-decoration: none; font-size: 14px; display: inline-block; margin-bottom: 20px; }}
        .back:hover {{ color: #111; }}
        h1 {{ font-size: 22px; font-weight: 600; color: #111; margin: 0 0 24px; }}
        .nb-card {{ display: flex; align-items: center; gap: 12px; padding: 14px 18px; border: 1px solid #e5e7eb; border-radius: 10px; background: white; margin-bottom: 8px; text-decoration: none; color: inherit; }}
        .nb-card:hover {{ border-color: #9ca3af; background: #f9fafb; }}
        .nb-num {{ font-size: 12px; color: #9ca3af; min-width: 36px; }}
        .nb-title {{ font-size: 14px; color: #111; }}
        .arrow {{ margin-left: auto; color: #9ca3af; }}
    </style>
</head>
<body>
    <a href="index.html" class="back">← Back to home</a>
    <h1>{topic['title']}</h1>
    {notebook_links}
</body>
</html>"""

        # Write this topic's landing page to docs folder
        with open(f"docs/{topic['id']}.html", "w", encoding="utf-8") as f:
            f.write(html)
        print(f"Built: {topic['id']}.html")

# ─── STEP 6: CONVERT NOTEBOOKS TO HTML ────────────────────────────────────────
# Uses nbconvert to turn each .ipynb into a self-contained HTML file

def convert_notebooks():
    os.makedirs("docs", exist_ok=True)  # create docs folder if it doesn't exist
    for topic in TOPICS:
        print(f"Converting {topic['title']}...")
        result = subprocess.run(
            f'jupyter nbconvert --to html --output-dir docs "{topic["folder"]}"/*.ipynb',
            shell=True, capture_output=True, text=True
        )
        print(result.stdout or result.stderr)

# ─── STEP 7: PUSH TO GITHUB ───────────────────────────────────────────────────
# Commits all changes and pushes to GitHub so your site updates

def push_to_github(message="update site"):
    for cmd in ["git add .", f'git commit -m "{message}"', "git push"]:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        print(r.stdout or r.stderr)

# ─── RUN EVERYTHING ───────────────────────────────────────────────────────────
print("Step 1: Embedding images...")
for topic in TOPICS:
    embed_images(topic)

print("\nStep 2: Building homepage...")
build_homepage()

print("\nStep 3: Building topic pages...")
build_topic_pages()

print("\nStep 4: Converting notebooks to HTML...")
convert_notebooks()

print("\nStep 5: Pushing to GitHub...")
push_to_github("update site with new structure")

print("\nDone! Visit: https://praveen-prabhu-bng.github.io/data-science-learning-journey/")